In [19]:
#import libraries
import subprocess
import sys
import os
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.preprocessing import LabelEncoder

In [38]:
OUTPUT_DIR = "../saved_models"

# Get the directory of the current script (.../notebooks/model/phases)
script_dir = os.path.dirname(os.path.abspath("__file__"))

# Move up two levels to the 'notebooks' directory (.../notebooks)
notebooks_dir = os.path.dirname(os.path.dirname(script_dir))

# Construct the correct absolute path
ALGORITHM_DIR = os.path.join(notebooks_dir, "model", "algorithms")

In [21]:
NORMAL_LABEL = "Benign"

In [22]:
def import_module_or_notebook(name, directory="."):
    py_path = os.path.join(directory, f"{name}.py")
    nb_path = os.path.join(directory, f"{name}.ipynb")

    # Determine if we need to convert the notebook
    needs_conversion = False

    if os.path.exists(nb_path):
        needs_conversion = True
    elif not os.path.exists(py_path):
        raise FileNotFoundError(f"Neither {name}.py nor {name}.ipynb found.")

    # Run conversion only when necessary
    if needs_conversion:
        subprocess.run(
            [
                sys.executable,
                "-m","nbconvert","--to","script","--output",
                name,nb_path,
            ],
            capture_output=True,
            check=True,
        )
        print(f"Converted and compiled {name}.ipynb")

    # Append directory to path and import
    if directory not in sys.path:
        sys.path.insert(0, directory)

    if name in sys.modules:
        mod = importlib.reload(sys.modules[name])
    else:
        mod = importlib.import_module(name)

    print(f"Done importing {name}")
    return mod

In [23]:
# Fetch dataset (packet data)
#PREPROCESS_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "preprocessing") 

#data_pipeline = import_module_or_notebook("phase2preprocess", PREPROCESS_DIR)

#packet_data, _, _ = data_pipeline

data = []

In [24]:
#Sample DATA
def get_training_data():
    X_raw, y_raw_int = make_classification(
        n_samples=12_000, n_features=84, n_informative=40,
        n_classes=8, n_clusters_per_class=1, random_state=42
    )
    ATTACK_CLASSES = ["Benign", "DDoS", "DoS", "Recon",
                      "Web-based", "BruteForce", "Spoofing", "Mirai"]
    y_raw = np.array([ATTACK_CLASSES[i] for i in y_raw_int])
    X_raw = pd.DataFrame(X_raw, columns=[f"feature_{i}" for i in range(84)])
    
    X_temp, X_test, y_temp, y_test = train_test_split(
        X_raw, y_raw, test_size=0.15, stratify=y_raw, random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.15, stratify=y_temp, random_state=42
    )
    
    NORMAL_LABEL = "Benign"

    return X_train, X_val, y_train, y_val

In [39]:
# import models to train on
def get_models():
    print(ALGORITHM_DIR)
    
    ALGORITHMS = [
        "model_isolation_forest",
        "model_autoencoder",
        "model_mlp"
    ]
    
    print("Importing notebooks...")
    models = {
        n: import_module_or_notebook(n, ALGORITHM_DIR) for n in ALGORITHMS
    }
    
    isolation_forest = models["model_isolation_forest"]
    autoencoder = models["model_autoencoder"]
    mlp = models["model_mlp"]

    return isolation_forest, autoencoder, mlp

In [40]:
# train model 1
def train_isolation_forest():
    X_train, X_val, y_train, y_val = get_training_data()
    isolation_forest, autoencoder, mlp = get_models()

    model, threshold, metrics = isolation_forest.train(
        X_train, y_train,
        X_val, y_val,
        normal_label=NORMAL_LABEL,
        model_kwargs= {
            "n_estimators":  200,
            "max_samples":   256,
            "contamination": "auto",
            "max_features":  1.0,
        },
        tune_threshold=True,
        save_path=os.path.join(OUTPUT_DIR, "isolation_forest.joblib")
    )

    return model, threshold, metrics

    

In [43]:
# train model 2
model, threshold, metrics = train_isolation_forest()

/Users/pauligbinedion/MASTERS/SUMMER/CAPSTONE/ECE597-Capstone-IoT-IDS/notebooks/model/algorithms
Importing notebooks...
Converted and compiled model_isolation_forest.ipynb
Done importing model_isolation_forest
Converted and compiled model_autoencoder.ipynb
Done importing model_autoencoder
Converted and compiled model_mlp.ipynb
Done importing model_mlp
Training on 1,080 benign samples (excluded 7,590 attack samples).
Isolation Forest fitted.
Threshold tuned: 0.08917  (val F1=0.9341)
Saved > ../saved_models/isolation_forest.joblib


In [ ]:
# train model 3

In [ ]:
#compare model metrics

In [ ]:
#save model(s)